In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import PIL
import tensorflow as tf
import opendatasets as od 
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
import pandas as pd
import os

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn import tree
import random
from PIL import Image
import cv2

from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout



In [2]:
# od.download(
#     "https://www.kaggle.com/datasets/andrewmvd/ocular-disease-recognition-odir5k")

data = pd.read_excel("Processed_data.xlsx")
data.head()
path_to_images = "./ocular-disease-recognition-odir5k/"

In [4]:
# Creating the target Labels 
def createTarget(df):
    df['Labels'] = [[0,0,0,0,0,0,0] for i in range (0,len(df))] #create a column for labels with 8 0s
    target_columns = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
    label = [] #create empty label array
    for i in range(0, len(df)):
        for target in target_columns:
            label.append(df.loc[i, target]) #append the value for the column for each row to the label
        
        df.at[i,'Labels'] = label #update the label column with the label array
        label = [] #reset label array
            

In [5]:
createTarget(data)

In [6]:
data = data.drop(['Patient ID', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O'], axis=1)
data

,Patient Age,Patient Sex,Filename,Diagnosis,Labels
0,69,Female,0_left.jpg,cataract,"[0, 0, 0, 1, 0, 0, 0, 0]"
1,57,Male,1_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
2,42,Male,2_left.jpg,"laser spot,moderate non proliferative retinopathy","[0, 1, 0, 0, 0, 0, 0, 1]"
3,66,Male,3_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
4,53,Male,4_left.jpg,macular epiretinal membrane,"[0, 0, 0, 0, 0, 0, 0, 1]"
...,...,...,...,...,...
6995,63,Male,4686_right.jpg,proliferative diabetic retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]"
6996,42,Male,4688_right.jpg,moderate non proliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]"
6997,54,Male,4689_right.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
6998,57,Male,4690_right.jpg,mild nonproliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]"


In [7]:
def binary_to_decimal(binary_list):
    """
    Converts a list of 0s and 1s (binary) to its decimal equivalent.

    :param binary_list: List of 0s and 1s.
    :return: Decimal equivalent of the binary number.
    """
    return sum(val * (2 ** idx) for idx, val in enumerate(reversed(binary_list)))

In [8]:
# Apply the function to each row
data['target'] = data['Labels'].apply(binary_to_decimal)
data

,Patient Age,Patient Sex,Filename,Diagnosis,Labels,target
0,69,Female,0_left.jpg,cataract,"[0, 0, 0, 1, 0, 0, 0, 0]",16
1,57,Male,1_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]",128
2,42,Male,2_left.jpg,"laser spot,moderate non proliferative retinopathy","[0, 1, 0, 0, 0, 0, 0, 1]",65
3,66,Male,3_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]",128
4,53,Male,4_left.jpg,macular epiretinal membrane,"[0, 0, 0, 0, 0, 0, 0, 1]",1
...,...,...,...,...,...,...
6995,63,Male,4686_right.jpg,proliferative diabetic retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]",64
6996,42,Male,4688_right.jpg,moderate non proliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]",64
6997,54,Male,4689_right.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]",128
6998,57,Male,4690_right.jpg,mild nonproliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]",64


In [9]:
# Path to your image dataset
image_dataset_path = "./ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Training images"
validation_path = "./ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Testing images"

In [10]:
data["filepath"] = data['Filename'].apply(lambda x: os.path.join(image_dataset_path, x))

In [11]:
import cv2
import numpy as np

# Function to preprocess images
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.resize(image, (128, 128))  # Resize to the desired input size for your CNN
    return image

# Applying the function to each image path
data['image'] = data['filepath'].apply(preprocess_image)

# Normalize age and encode gender
data['Patient Age'] = data['Patient Age'] / 100  # Assuming age is less than 100
data['Patient Sex'] = data['Patient Sex'].map({'Male': 0, 'Female': 1})  # Binary encoding

In [13]:
data

,Patient Age,Patient Sex,Filename,Diagnosis,Labels,target,filepath,image
0,0.69,1,0_left.jpg,cataract,"[0, 0, 0, 1, 0, 0, 0, 0]",16,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
1,0.57,0,1_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]",128,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
2,0.42,0,2_left.jpg,"laser spot,moderate non proliferative retinopathy","[0, 1, 0, 0, 0, 0, 0, 1]",65,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
3,0.66,0,3_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]",128,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
4,0.53,0,4_left.jpg,macular epiretinal membrane,"[0, 0, 0, 0, 0, 0, 0, 1]",1,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
...,...,...,...,...,...,...,...,...
6995,0.63,0,4686_right.jpg,proliferative diabetic retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]",64,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
6996,0.42,0,4688_right.jpg,moderate non proliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]",64,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
6997,0.54,0,4689_right.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]",128,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."
6998,0.57,0,4690_right.jpg,mild nonproliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]",64,./ocular-disease-recognition-odir5k/ODIR-5K/OD...,"[[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], ..."


In [12]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np


# Example DataFrame
# df = pd.DataFrame({'age': [25, 30, 45], 'gender': ['male', 'female', 'male']})

# Scale age
scaler = StandardScaler()
ages_scaled = scaler.fit_transform(data[['Patient Age']])

# Encode gender
encoder = OneHotEncoder()
genders_encoded = encoder.fit_transform(data[['Patient Sex']]).toarray()

# Combine age and gender
age_gender_combined = np.hstack((ages_scaled, genders_encoded))

# Now, age_gender_combined is what you will use as train_age_gender in model training


In [13]:
# import tensorflow as tf
# from tensorflow.keras import layers, models
# from tensorflow.keras.applications.vgg19 import VGG19

# vgg = VGG19(weights="imagenet",include_top = False,input_shape=(128,128,3))
# # Assuming image_matrix is a 4D tensor: [samples, height, width, channels]
# # age and gender are 1D tensors

# def create_model():
#     # Image input branch - CNN
# #     image_input = layers.Input(shape=(128, 128, 3))
    
#     image_input = layers.Input(shape=(128, 128, 3))
#     x = vgg(image_input)
#     x = layers.Flatten()(x)
#     image_branch = models.Model(inputs=image_input, outputs=x)

# #     x = layers.Conv2D(32, (3, 3), activation='relu')(image_input)
# #     x = layers.MaxPooling2D((2, 2))(x)
# #     x = layers.Flatten()(x)
# #     image_branch = models.Model(inputs=image_input, outputs=x)

#     # Age and gender input branch
#     age_gender_input = layers.Input(shape=(3,))  # Assuming age and gender are scaled/encoded
#     y = layers.Dense(32, activation='relu')(age_gender_input)
#     age_gender_branch = models.Model(inputs=age_gender_input, outputs=y)

#     # Concatenate branches
#     combined = layers.concatenate([image_branch.output, age_gender_branch.output])

#     # Dense layers and output
#     z = layers.Dense(64, activation='relu')(combined)
#     z = layers.Dense(8, activation='sigmoid')(z)  # num_diseases = length of label list

#     model = models.Model(inputs=[image_branch.input, age_gender_branch.input], outputs=z)

#     return model


# model = create_model()
# model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])




# import tensorflow as tf
# from tensorflow.keras import layers, models
# from tensorflow.keras.applications.vgg19 import VGG19

# # Load the VGG19 model pre-trained on ImageNet data
# vgg = VGG19(weights="imagenet", include_top=False, input_shape=(128, 128, 3))

# def create_model():
#     # Image input branch - CNN
#     image_input = layers.Input(shape=(128, 128, 3))
#     x = vgg(image_input)  # Pass the input through the VGG19 model
#     x = layers.Flatten()(x)  # Flatten the output of VGG19
#     image_branch = models.Model(inputs=image_input, outputs=x)

#     # Dense layers and output
#     z = layers.Dense(64, activation='relu')(image_branch.output)
#     z = layers.Dense(8, activation='sigmoid')(z)  # num_diseases = length of label list

#     # Create the model with only image input
#     model = models.Model(inputs=image_input, outputs=z)

#     return model

# # Create and compile the model
# model = create_model()
# model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# import tensorflow as tf
# from tensorflow.keras import layers, models
# from tensorflow.keras.applications.vgg19 import VGG19

# vgg = VGG19(weights="imagenet",include_top = False,input_shape=(128,128,3))
# # Assuming image_matrix is a 4D tensor: [samples, height, width, channels]
# # age and gender are 1D tensors

# def create_model():
#     # Image input branch - CNN
# #     image_input = layers.Input(shape=(128, 128, 3))
    
#     image_input = layers.Input(shape=(128, 128, 3))
#     x = vgg(image_input)
#     x = layers.Flatten()(x)
#     image_branch = models.Model(inputs=image_input, outputs=x)

# #     x = layers.Conv2D(32, (3, 3), activation='relu')(image_input)
# #     x = layers.MaxPooling2D((2, 2))(x)
# #     x = layers.Flatten()(x)
# #     image_branch = models.Model(inputs=image_input, outputs=x)

#     # Age and gender input branch
#     age_gender_input = layers.Input(shape=(3,))  # Assuming age and gender are scaled/encoded
#     y = layers.Dense(32, activation='relu')(age_gender_input)
#     age_gender_branch = models.Model(inputs=age_gender_input, outputs=y)

#     # Concatenate branches
#     combined = layers.concatenate([image_branch.output, age_gender_branch.output])

#     # Dense layers and output
#     z = layers.Dense(64, activation='relu')(combined)
#     z = layers.Dense(8, activation='sigmoid')(z)  # num_diseases = length of label list

#     model = models.Model(inputs=[image_branch.input, age_gender_branch.input], outputs=z)

#     return model


# model = create_model()
# model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

from tensorflow.keras.applications.vgg19 import VGG19
import tensorflow as tf
vgg = VGG19(weights="imagenet",include_top = False,input_shape=(128,128,3))

for layer in vgg.layers:
    layer.trainable = False

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Flatten, Dense, Input
# image_input = Input(shape=(128, 128, 3))
model = Sequential()
model.add(vgg)
model.add(Flatten())
model.add(Dense(256,activation = "relu"))
model.add(tf.keras.layers.BatchNormalization())
model.add(Dense(256,activation = "relu"))
model.add(tf.keras.layers.BatchNormalization())
model.add(Dense(39,activation="softmax"))

model.summary()



80134624/80134624 [==============================] - 7s 0us/step
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg19 (Functional)          (None, 4, 4, 512)         20024384  
                                                                 
 flatten (Flatten)           (None, 8192)              0         
                                                                 
 dense (Dense)               (None, 256)               2097408   
                                                                 
 batch_normalization (Batch  (None, 256)               1024      
 Normalization)                                                  
                                                                 
 dense_1 (Dense)             (None, 256)               65792     
                                                                 
 batch_normalization_1 (Bat  (None, 256)               

In [15]:
image_data = np.array(data["image"].tolist())

labels = data["target"]

In [16]:
len(labels)

7000

In [17]:
len(image_data)

7000

In [18]:
len(labels.unique())

39

In [19]:
# train_images, val_images, train_age_gender, val_age_gender, train_labels, val_labels = train_test_split(
#     image_data, age_gender_combined, labels, test_size=0.2, random_state=42)

train_images, val_images, train_labels, val_labels = train_test_split(
    image_data, labels, test_size=0.2, random_state=42)

print(len(train_images))
print(len(val_images))
print(len(train_labels))
print(len(val_labels))

5600
1400
5600
1400


In [20]:
# train_labels = np.array(train_labels.tolist())
# val_labels = np.array(val_labels.tolist())

#model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])
# Training - assuming you have prepared `train_data`, `train_labels`, `val_data`, `val_labels`
# model.fit([train_images, train_age_gender], train_labels, validation_data=([val_images, val_age_gender], val_labels), epochs=10, batch_size=32)

#model.fit(train_images, train_labels, validation_data=(val_images, val_labels), epochs=10, batch_size=32)

# Assuming train_labels and val_labels are your label arrays
# and num_classes is the number of classes (39 in this case)
from tensorflow.keras.utils import to_categorical

num_classes = 39  # Update this if the number of classes is different
label_encoder = LabelEncoder()
print(len(train_labels))
train_labels = label_encoder.fit_transform(train_labels)
train_labels_one_hot = to_categorical(train_labels, num_classes)

val_labels = label_encoder.fit_transform(val_labels)
val_labels_one_hot = to_categorical(val_labels, num_classes)

len(train_labels)
len(train_images)

5600


5600

In [23]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [22]:

model.fit(train_images, train_labels_one_hot, validation_data=(val_images, val_labels_one_hot), epochs=10, batch_size=32)


Epoch 1/10


175/175 [==============================] - 212s 1s/step - loss: 2.6616 - accuracy: 0.3805 - val_loss: 4.0198 - val_accuracy: 0.0550
Epoch 2/10
175/175 [==============================] - 200s 1s/step - loss: 1.5572 - accuracy: 0.5059 - val_loss: 5.0808 - val_accuracy: 0.0486
Epoch 3/10
175/175 [==============================] - 179s 1s/step - loss: 1.3253 - accuracy: 0.5559 - val_loss: 6.1375 - val_accuracy: 0.0779
Epoch 4/10
175/175 [==============================] - 210s 1s/step - loss: 1.1939 - accuracy: 0.5948 - val_loss: 8.8057 - val_accuracy: 0.0314
Epoch 5/10
175/175 [==============================] - 177s 1s/step - loss: 1.0927 - accuracy: 0.6136 - val_loss: 7.9061 - val_accuracy: 0.0321
Epoch 6/10
175/175 [==============================] - 173s 992ms/step - loss: 1.0051 - accuracy: 0.6368 - val_loss: 6.6513 - val_accuracy: 0.0521
Epoch 7/10
175/175 [==============================] - 175s 1s/step - loss: 0.8639 - accuracy: 0.6966 - val_loss: 8.1603 - val_accuracy: 0